# Boundary-driven XXZ: MPDO tutorial

The saved results use Julia 1.13.1. The calculation uses a fused ket/bra ITensorMPS representation, an explicit MPO view, and an independent dense Lindblad reference. Run from the package directory with `julia --project=.`.

## Evolve a four-site chain

The positive current direction is left to right. At finite time the currents need not yet be uniform.

In [1]:
using BoundaryDrivenXXZ, LinearAlgebra, Printf
m = Model(4; J=0.5, delta=0.3, gamma=0.5, mu=0.5)
s = evolve(m, 0.4; dt=0.02, maxdim=64)
o = observables(m,s)
@printf("trace = %.9f\n",real(o.trace))
println("magnetizations = ",round.(o.magnetization;digits=6))
println("currents = ",round.(o.current;digits=6))
println("MPO type = ",typeof(density_mpo(m,s)))

trace = 0.999999999
magnetizations = [0.15686, 0.008394, -0.008394, -0.15686]
currents = [0.061257, 0.005241, 0.061256]
MPO type = ITensorMPS.MPO


## Check full density matrices

The dense solver builds the global Hamiltonian and four edge jumps independently of the local Liouvillian gates.

In [2]:
for n in 2:4
    model = Model(n)
    state = evolve(model,0.4;dt=0.02,cutoff=1e-14,maxdim=256)
    rho = density_matrix(model,state)
    ref = dense_evolve(model,0.4)
    @printf("N=%d max entry error %.3e, trace %.9f, min eigenvalue %.6f\n",n,maximum(abs.(rho-ref)),real(tr(rho)),minimum(eigvals(Hermitian((rho+adjoint(rho))/2))))
end

N=2 max entry error 4.627e-09, trace 1.000000000, min eigenvalue 0.176184
N=3 max entry error 1.970e-06, trace 1.000000000, min eigenvalue 0.087559
N=4 max entry error 7.522e-07, trace 1.000000002, min eigenvalue 0.043550


## Current continuity and late-time density

At stationarity the two currents equal the left source and the negative right source. The final density is compared with a trace-constrained dense NESS.

In [3]:
m3 = Model(3)
late = evolve(m3,12.0;dt=0.05,maxdim=16)
ol = observables(m3,late)
@printf("currents = %s, left source = %.6f, right source = %.6f\n",string(round.(ol.current;digits=6)),ol.left_source,ol.right_source)
@printf("max NESS density error = %.3e\n",maximum(abs.(density_matrix(m3,late)-dense_steady_state(m3))))

currents = [0.38279, 0.382643], left source = 0.382704, right source = -0.382718
max NESS density error = 1.568e-05


## Step and bond refinement

The local gate split is second order. A small bond cap gives a distinct compression error.

In [4]:
ref = dense_evolve(m3,0.5)
for (step, bond) in ((0.1,16),(0.025,16),(0.025,2))
    approx = density_matrix(m3,evolve(m3,0.5;dt=step,maxdim=bond))
    @printf("dt=%.3f maxdim=%d Frobenius error=%.3e\n",step,bond,norm(approx-ref))
end

dt=0.100 maxdim=16 Frobenius error=1.263e-04
dt=0.025 maxdim=16 Frobenius error=7.904e-06
dt=0.025 maxdim=2 Frobenius error=2.424e-02
